# Phase 5 — Analyse comparative

Les trois portefeuilles tiennent les mêmes titres, sur les mêmes mois, avec les mêmes rendements en CHF et le même SARON. Le seul changement d'une colonne à l'autre est la règle de poids.

Ce notebook lit les CSV déjà produits. Il ne recalcule pas les poids. Les formules sont dans `documentation/05_portfolio_construction.ipynb`. Le détail arithmétique est dans `reports/portfolio/05_calculs_detailles.ipynb`.

Les chiffres des sections suivantes sont imprimés depuis les CSV de cette exécution. Le poids d'un trimestre est fixé avec les soixante mois précédents, sous les plafonds de l'IPS. La période suivie s'arrête au dernier mois du SARON.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import MARKET_HISTORY_YEARS, REPORTING_CURRENCY

from src.portfolio_construction.build_portfolios import (
    COMPARISON_FILE,
    METRICS_FILE,
    MONTHLY_RETURNS_FILE,
    WEIGHTS_FILE,
)

metrics = pd.read_csv(METRICS_FILE)
comparison = pd.read_csv(COMPARISON_FILE)
monthly = pd.read_csv(MONTHLY_RETURNS_FILE)
weights = pd.read_csv(WEIGHTS_FILE)

labels = {
    "equal_weight": "Équipondéré",
    "minimum_volatility": "Volatilité minimale",
    "maximum_sharpe": "Sharpe maximal",
}
table = metrics.set_index("strategy").loc[list(labels)]

def fr(value, digits):
    return f"{value:.{digits}f}".replace(".", ",")


def fr_pct(value, digits=1):
    return fr(float(value) * 100, digits) + " %"


def final_wealth(strategy):
    series = monthly.loc[monthly["strategy"].eq(strategy), "portfolio_return"]
    return float((1 + series).prod())


def above_count(strategy, threshold=0.01):
    group = weights.loc[weights["strategy"].eq(strategy)]
    return int(group["weight"].ge(threshold).sum())


def top_holding(strategy):
    group = weights.loc[weights["strategy"].eq(strategy)].nlargest(1, "weight")
    row = group.iloc[0]
    return str(row["name"]), float(row["weight"])


def following_names(strategy, count=4):
    group = weights.loc[weights["strategy"].eq(strategy)].nlargest(count + 1, "weight")
    return ", ".join(group["name"].astype(str).iloc[1:].tolist())


def describe_equal_weight():
    row = table.loc["equal_weight"]
    n = int(row["names"])
    print(
        "\n".join(
            [
                f"Chaque titre pèse 1/{n}, soit {fr_pct(1 / n)}.",
                f"Le nombre effectif de titres est {fr(row['effective_holdings'], 2)}.",
                (
                    f"Sur {row['window_start']} – {row['window_end']}, "
                    f"soit {int(row['months'])} mois, ce portefeuille rend "
                    f"{fr_pct(row['annualized_return'])} par an."
                ),
                f"Sa volatilité annualisée est {fr_pct(row['annualized_volatility'])}.",
                f"Son Sharpe, après SARON, est {fr(row['sharpe_ratio'], 2)}.",
                (
                    f"Un placement de 1 CHF vaut {fr(final_wealth('equal_weight'), 2)} CHF "
                    "en fin de période."
                ),
                f"Le drawdown maximal est de {fr_pct(-row['maximum_drawdown'])}.",
                "Ces chiffres servent de base de comparaison dans cette note.",
            ]
        )
    )


def describe_minimum_volatility():
    base = table.loc["equal_weight"]
    row = table.loc["minimum_volatility"]
    name, weight = top_holding("minimum_volatility")
    gap = (float(base["annualized_volatility"]) - float(row["annualized_volatility"])) * 100
    if gap >= 0:
        gap_text = f"{fr(gap, 1)} points de pourcentage sous l'équipondéré"
    else:
        gap_text = f"{fr(-gap, 1)} points de pourcentage au-dessus de l'équipondéré"
    print(
        "\n".join(
            [
                f"La volatilité annualisée est {fr_pct(row['annualized_volatility'])}, soit {gap_text}.",
                f"Le nombre effectif de titres est {fr(row['effective_holdings'], 1)}.",
                f"{above_count('minimum_volatility')} titres dépassent 1 %.",
                f"Le plus gros poids est {name}, à {fr_pct(weight)}.",
                f"Les poids suivants sont {following_names('minimum_volatility')}.",
                (
                    f"Le rendement annualisé est {fr_pct(row['annualized_return'])}. "
                    f"Le Sharpe est {fr(row['sharpe_ratio'], 2)}, "
                    f"contre {fr(base['sharpe_ratio'], 2)} pour l'équipondéré."
                ),
                (
                    f"Le drawdown maximal est {fr_pct(-row['maximum_drawdown'])}, "
                    f"contre {fr_pct(-base['maximum_drawdown'])} pour l'équipondéré."
                ),
                f"La performance cumulée est {fr(final_wealth('minimum_volatility'), 2)}.",
            ]
        )
    )


def describe_maximum_sharpe():
    base = table.loc["equal_weight"]
    low = table.loc["minimum_volatility"]
    row = table.loc["maximum_sharpe"]
    name, weight = top_holding("maximum_sharpe")
    print(
        "\n".join(
            [
                (
                    f"Sur {row['window_start']} – {row['window_end']}, ce rapport vaut "
                    f"{fr(row['sharpe_ratio'], 2)}, contre {fr(base['sharpe_ratio'], 2)} "
                    f"pour l'équipondéré et {fr(low['sharpe_ratio'], 2)} pour la volatilité minimale."
                ),
                (
                    f"Le rendement annualisé est {fr_pct(row['annualized_return'])}. "
                    f"La volatilité annualisée est {fr_pct(row['annualized_volatility'])}, "
                    f"contre {fr_pct(base['annualized_volatility'])} pour l'équipondéré."
                ),
                (
                    f"Le drawdown maximal est {fr_pct(-row['maximum_drawdown'])}, "
                    f"contre {fr_pct(-base['maximum_drawdown'])} pour l'équipondéré."
                ),
                f"Le nombre effectif de titres est {fr(row['effective_holdings'], 1)}.",
                f"{above_count('maximum_sharpe')} titres dépassent 1 %.",
                f"Le plus gros poids est {name}, à {fr_pct(weight)}, puis {following_names('maximum_sharpe')}.",
                f"Un placement de 1 CHF vaut {fr(final_wealth('maximum_sharpe'), 2)} CHF en fin de période.",
                (
                    "Ces poids ont été choisis avec les soixante mois précédents, "
                    "sous les plafonds de l'IPS. "
                    "Le ratio de Sharpe publié est celui du chemin qui en résulte."
                ),
            ]
        )
    )


def describe_wealth():
    row = table.loc["equal_weight"]
    print(
        f"L'indice part de 1 avant {row['window_start']}. "
        f"Les fins d'année sont gardées, ainsi que {row['window_end']}, dernier mois de la période."
    )
    for strategy, label in labels.items():
        print(f"Performance cumulée, {label} : {fr(final_wealth(strategy), 2)}.")
    print("Cet écart de performance cumulée décrit le chemin des poids décidés chaque trimestre, puis laissés dériver.")


def describe_overview():
    base = table.loc["equal_weight"]
    low = table.loc["minimum_volatility"]
    high = table.loc["maximum_sharpe"]
    print(
        "\n".join(
            [
                (
                    f"L'équipondéré couvre les {int(base['names'])} titres. "
                    f"Son Sharpe est {fr(base['sharpe_ratio'], 2)} et sa volatilité "
                    f"{fr_pct(base['annualized_volatility'])}."
                ),
                (
                    f"La volatilité minimale a {fr(low['effective_holdings'], 1)} titres effectifs "
                    f"et une volatilité de {fr_pct(low['annualized_volatility'])}. "
                    f"Son Sharpe est {fr(low['sharpe_ratio'], 2)}."
                ),
                (
                    f"Le Sharpe maximal atteint {fr(high['sharpe_ratio'], 2)}, "
                    f"avec une volatilité de {fr_pct(high['annualized_volatility'])} "
                    f"et une performance cumulée de {fr(final_wealth('maximum_sharpe'), 2)}."
                ),
                (
                    "Ces chiffres décrivent le chemin publié. "
                    "Ils ne décrivent pas l'échantillon de soixante mois qui a servi à choisir un poids."
                ),
            ]
        )
    )


## 1. Les trois résultats

Le rendement annualisé est le taux géométrique des mois de la période. La volatilité est celle des rendements du portefeuille. Le Sharpe retire le SARON mensuel avant de diviser par le risque. Le drawdown maximal est négatif : il mesure le repli le plus profond par rapport au plus haut atteint. Le nombre effectif de titres vaut le nombre de lignes lorsque tous les poids sont égaux, et il baisse lorsque les poids se concentrent.


In [2]:
shown = table.copy()
shown.index = shown.index.map(labels)
for column in ("annualized_return", "annualized_volatility", "maximum_drawdown", "maximum_weight"):
    shown[column] = (shown[column] * 100).map(lambda value: f"{value:.1f} %")
shown["sharpe_ratio"] = shown["sharpe_ratio"].map(lambda value: f"{value:.2f}")
shown["effective_holdings"] = shown["effective_holdings"].map(lambda value: f"{value:.1f}")
shown.loc[:, [
    "names",
    "months",
    "window_start",
    "window_end",
    "annualized_return",
    "annualized_volatility",
    "sharpe_ratio",
    "maximum_drawdown",
    "maximum_weight",
    "effective_holdings",
]].rename(columns={
    "names": "Titres",
    "months": "Mois",
    "window_start": "Début",
    "window_end": "Fin",
    "annualized_return": "Rendement annualisé",
    "annualized_volatility": "Volatilité annualisée",
    "sharpe_ratio": "Sharpe",
    "maximum_drawdown": "Drawdown maximal",
    "maximum_weight": "Poids maximum",
    "effective_holdings": "Titres effectifs",
})


,Titres,Mois,Début,Fin,Rendement annualisé,Volatilité annualisée,Sharpe,Drawdown maximal,Poids maximum,Titres effectifs
strategy,,,,,,,,,,
Équipondéré,28,120,2016-11,2026-10,8.0 %,12.3 %,0.69,-22.9 %,3.6 %,28.0
Volatilité minimale,28,120,2016-11,2026-10,6.3 %,11.8 %,0.59,-23.1 %,5.0 %,22.7
Sharpe maximal,28,120,2016-11,2026-10,6.3 %,12.1 %,0.57,-27.4 %,5.0 %,22.6


## 2. Diversification simple

L'équipondéré répond à une question précise : que se passe-t-il si aucun titre n'est préféré ? Ses poids ne lisent pas les rendements passés.


In [3]:
describe_equal_weight()


Chaque titre pèse 1/28, soit 3,6 %.
Le nombre effectif de titres est 28,00.
Sur 2016-11 – 2026-10, soit 120 mois, ce portefeuille rend 8,0 % par an.
Sa volatilité annualisée est 12,3 %.
Son Sharpe, après SARON, est 0,69.
Un placement de 1 CHF vaut 2,15 CHF en fin de période.
Le drawdown maximal est de 22,9 %.
Ces chiffres servent de base de comparaison dans cette note.


## 3. Réduction du risque

La volatilité minimale cherche la plus petite variance mensuelle. Les poids viennent de la covariance des rendements des soixante mois qui précèdent chaque trimestre.


In [4]:
describe_minimum_volatility()


La volatilité annualisée est 11,8 %, soit 0,5 points de pourcentage sous l'équipondéré.
Le nombre effectif de titres est 22,7.
2893 titres dépassent 1 %.
Le plus gros poids est CHINA LONGYUAN POWER GROUP H, à 7,1 %.
Les poids suivants sont DAIICHI SANKYO LTD, GENMAB, VESTAS WIND SYSTEMS, CHINA LONGYUAN POWER GROUP H.
Le rendement annualisé est 6,3 %. Le Sharpe est 0,59, contre 0,69 pour l'équipondéré.
Le drawdown maximal est 23,1 %, contre 22,9 % pour l'équipondéré.
La performance cumulée est 1,84.


In [5]:
group = weights.loc[weights["strategy"].eq("minimum_volatility")].nlargest(8, "weight")
group.assign(weight=(group["weight"] * 100).map(lambda value: f"{value:.1f} %")).loc[
    :, ["yahoo_symbol", "name", "listing_currency", "weight"]
].rename(columns={
    "yahoo_symbol": "Symbole",
    "name": "Titre",
    "listing_currency": "Devise",
    "weight": "Poids",
})


,Symbole,Titre,Devise,Poids
4816,0916.HK,CHINA LONGYUAN POWER GROUP H,HKD,7.1 %
3808,4568.T,DAIICHI SANKYO LTD,JPY,6.9 %
4564,GMAB.CO,GENMAB,DKK,6.8 %
4648,VWS.CO,VESTAS WIND SYSTEMS,DKK,6.8 %
4788,0916.HK,CHINA LONGYUAN POWER GROUP H,HKD,6.8 %
5068,3898.HK,ZHUZHOU CRRC TIMES ELECTRIC LTD H,HKD,6.8 %
3780,4568.T,DAIICHI SANKYO LTD,JPY,6.4 %
4956,3898.HK,ZHUZHOU CRRC TIMES ELECTRIC LTD H,HKD,6.3 %


## 4. Rendement ajusté du risque

Le Sharpe maximal cherche le plus grand rapport entre l'excédent au-dessus du SARON et l'écart-type de cet excédent. L'équipondéré est le seul des trois dont les poids ne dépendent pas de ces rendements.


In [6]:
describe_maximum_sharpe()


Sur 2016-11 – 2026-10, ce rapport vaut 0,57, contre 0,69 pour l'équipondéré et 0,59 pour la volatilité minimale.
Le rendement annualisé est 6,3 %. La volatilité annualisée est 12,1 %, contre 12,3 % pour l'équipondéré.
Le drawdown maximal est 27,4 %, contre 22,9 % pour l'équipondéré.
Le nombre effectif de titres est 22,6.
2913 titres dépassent 1 %.
Le plus gros poids est ALNYLAM PHARMACEUTICALS, à 7,0 %, puis VESTAS WIND SYSTEMS, DAIICHI SANKYO LTD, ALNYLAM PHARMACEUTICALS, TCL ZHONGHUAN RENEWABLE ENERGY TEC.
Un placement de 1 CHF vaut 1,83 CHF en fin de période.
Ces poids ont été choisis avec les soixante mois précédents, sous les plafonds de l'IPS. Le ratio de Sharpe publié est celui du chemin qui en résulte.


In [7]:
group = weights.loc[weights["strategy"].eq("maximum_sharpe")].nlargest(8, "weight")
group.assign(weight=(group["weight"] * 100).map(lambda value: f"{value:.1f} %")).loc[
    :, ["yahoo_symbol", "name", "listing_currency", "weight"]
].rename(columns={
    "yahoo_symbol": "Symbole",
    "name": "Titre",
    "listing_currency": "Devise",
    "weight": "Poids",
})


,Symbole,Titre,Devise,Poids
8680,ALNY,ALNYLAM PHARMACEUTICALS,USD,7.0 %
8008,VWS.CO,VESTAS WIND SYSTEMS,DKK,6.9 %
7168,4568.T,DAIICHI SANKYO LTD,JPY,6.6 %
9688,ALNY,ALNYLAM PHARMACEUTICALS,USD,6.6 %
8344,002129.SZ,TCL ZHONGHUAN RENEWABLE ENERGY TEC,CNY,6.5 %
7924,GMAB.CO,GENMAB,DKK,6.5 %
7756,4523.T,EISAI,JPY,6.4 %
7728,4523.T,EISAI,JPY,6.4 %


## 5. Performance cumulée

Chaque ligne est la performance cumulée en fin de mois, après capitalisation des rendements mensuels.


In [8]:
describe_wealth()


L'indice part de 1 avant 2016-11. Les fins d'année sont gardées, ainsi que 2026-10, dernier mois de la période.
Performance cumulée, Équipondéré : 2,15.
Performance cumulée, Volatilité minimale : 1,84.
Performance cumulée, Sharpe maximal : 1,83.
Cet écart de performance cumulée décrit le chemin des poids décidés chaque trimestre, puis laissés dériver.


In [9]:
wealth = monthly.loc[:, ["month", "strategy", "portfolio_return"]].copy()
wealth["wealth"] = wealth.groupby("strategy")["portfolio_return"].transform(
    lambda series: (1 + series).cumprod()
)
marks = wealth["month"].str.endswith("-12") | wealth["month"].eq(wealth["month"].max())
wealth_table = wealth.loc[marks].pivot(index="month", columns="strategy", values="wealth")
wealth_table = wealth_table.loc[:, list(labels)]
wealth_table.columns = [labels[column] for column in wealth_table.columns]
wealth_table.round(2)


,Équipondéré,Volatilité minimale,Sharpe maximal
month,,,
2016-12,1.01,1.00,1.01
2017-12,1.26,1.19,1.20
2018-12,1.27,1.25,1.24
2019-12,1.55,1.51,1.53
2020-12,1.71,1.59,1.62
2021-12,2.04,1.87,1.89
2022-12,1.85,1.71,1.69
2023-12,1.76,1.54,1.54
2024-12,1.85,1.70,1.67


## 6. Lecture d'ensemble

Les trois objectifs ne désignent pas le même portefeuille.


In [10]:
describe_overview()


L'équipondéré couvre les 28 titres. Son Sharpe est 0,69 et sa volatilité 12,3 %.
La volatilité minimale a 22,7 titres effectifs et une volatilité de 11,8 %. Son Sharpe est 0,59.
Le Sharpe maximal atteint 0,57, avec une volatilité de 12,1 % et une performance cumulée de 1,83.
Ces chiffres décrivent le chemin publié. Ils ne décrivent pas l'échantillon de soixante mois qui a servi à choisir un poids.
